In [2]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [4]:
from rag_helper import RAGBase
from ingest import load_faq_data, build_index

import os
from dotenv import load_dotenv
load_dotenv()


from openai import OpenAI
openai_client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

model = 'openai/gpt-oss-20b'
documents = load_faq_data()
index = build_index(documents)


In [7]:
def search(query):

    """
    Search the FAQ database for entries matching the given query.
    """

    boost_dict = {'question': 3.0, 'section': 0.5}
    filter_dict = {'course': 'llm-zoomcamp'}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

## Registering the tool

In [ ]:
agent_tools = Tools()

# don't need to define the schema for the search tool
# toykit will automatically infer the schema from the function signature and docstring
agent_tools.add_tool(search)

In [9]:
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the FAQ database for entries matching the given query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [25]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.

CRITICAL FOR TOOL USAGE:
When calling a function/tool, your JSON arguments must be perfectly valid. 
Do NOT include trailing commas before closing braces. 
Example of BAD JSON: {"query": "something",}
Example of GOOD JSON: {"query": "something"}

"""

question = 'I just discovered the course. Can I join it?'



In [26]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt = instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(
        model=model,
        client=OpenAI(
            base_url="https://api.groq.com/openai/v1",
            api_key=os.getenv("GROQ_API_KEY")
        ))
)

In [27]:
result   = runner.loop(
    prompt = "How do I run Olama locally",
    callback=callback
)

-> Response received


-> Response received


OS,Install method
macOS,Download the .pkg from https://ollama.com/download and run the installer.
Windows,Download the .msi from https://ollama.com/download and run the installer.
Issue,Fix
ollama: command not found,"Ensure the installation directory (e.g., ~/.local/bin) is in your PATH."
Model download stuck or slow,Check your internet connection and that you have sufficient disk space (≈5 GB free).
“exec: ‘streamlit’: file not found” when running in Docker,"If you want a Docker deployment, add streamlit to the CMD/ENTRYPOINT of your Dockerfile (not covered here, but see the Docker FAQ)."


In [28]:
result.cost

In [29]:
result.all_messages

[EasyInputMessage(content='\nYou\'re a course teaching assistant.\nYou\'re given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches.\n\nTry to expand your search by using new keywords\nbased on the results you get from the search.\n\nAt the end, ask if there are other areas that the user wants to explore.\n\nCRITICAL FOR TOOL USAGE:\nWhen calling a function/tool, your JSON arguments must be perfectly valid. \nDo NOT include trailing commas before closing braces. \nExample of BAD JSON: {"query": "something",}\nExample of GOOD JSON: {"query": "something"}\n\n', role='developer', phase=None, type=None),
 EasyInputMessage(content='How do I run Olama locally', role='user', phase=None, type=None),
 ResponseReasoningItem(id='resp_01kwbzfk4hesd9zgav7tnew2tn', summary=[], type='reasoning', content=[Content(te

Role: Course Teaching Assistant

Task: Answer student questions. Expand search terms using keywords from the user's question, execute multiple searches, and ask if there are other areas to explore at the end.
🙋‍♂️ Student Question

    "How do I run Olama locally"

🔍 Step 1: Initial Search & Execution

The system recognized that "Olama" was a typo for Ollama (the local LLM runner) and executed a search.

    Query: "Ollama run locally instructions"

    Key Search Results Retrieved:

        Installation: Download links for macOS (.pkg), Windows (.msi), and Linux command (curl -fsSL [https://ollama.com/install.sh](https://ollama.com/install.sh) | sh).

        CLI Execution: Run ollama run llama3 to download and chat.

        Testing: Run curl http://localhost:11434 to test the local server.

        Python Client: pip install ollama and a quick ollama.chat() code snippet.

🔍 Step 2: Search Expansion (Following Instructions)

To ensure no edge cases or troubleshooting details were missed, the assistant expanded its search.

    Query: "Ollama local run steps"

    Additional Context Retrieved:

        Encountering a Connection refused error can be resolved by launching/restarting the background daemon using ollama serve.

📋 Final Formatted Output Sent to the Student

In [30]:
# Continue the conversation with follow-up questions
result2 = runner.loop(
    prompt="How do I run a different model?",
    previous_messages=result.all_messages,
    callback=callback,
)

-> Response received


-> Response received


In [31]:
# interactive chat
runner.run()

-> Response received


-> Response received


-> Response received


Why it’s popular,Typical usage scenario,Example projects / repos
Zero‑config local LLM,"Run a model locally without cloud APIs, preserving privacy and eliminating latency",ollama run llama3 → instant chat in terminal
Lightweight Docker,Quick spin‑up for demos or CI pipelines,Docker Hub image ollama/ollama used in CI to test prompts
Python SDK,Embed LLM logic directly into apps,pip install ollama + ollama.chat() in a Streamlit or FastAPI backend
Notebook integration,Test prompts or build prototypes in Jupyter,!curl or ollama shell calls in a cell
Multiple models,"Switch between LLaMA‑3, Mistral‑7B, etc., on the same machine",ollama pull mistralai/mistral-7b + ollama run mistralai/mistral-7b
Rapid iteration,Quickly test changes to prompts or system messages,ollama serve → tweak prompt → run again without redeploying


Chat ended.


/home/hsu/Documents/Learning/llm-zoomcamp-cod/.venv/lib/python3.10/site-packages/toyaikit/chat/runners.py:184: UnknownModelWarning: No pricing data for model 'openai/gpt-oss-20b'. Register it with PricingConfig.register_model(...) to get cost calculations.
  combined_cost = self.pricing_config.calculate_cost(


LoopResult(new_messages=[EasyInputMessage(content='\nYou\'re a course teaching assistant.\nYou\'re given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches.\n\nTry to expand your search by using new keywords\nbased on the results you get from the search.\n\nAt the end, ask if there are other areas that the user wants to explore.\n\nCRITICAL FOR TOOL USAGE:\nWhen calling a function/tool, your JSON arguments must be perfectly valid. \nDo NOT include trailing commas before closing braces. \nExample of BAD JSON: {"query": "something",}\nExample of GOOD JSON: {"query": "something"}\n\n', role='developer', phase=None, type=None), EasyInputMessage(content='use Ollama in ubuntu', role='user', phase=None, type=None), ResponseReasoningItem(id='resp_01kwbzg9j9f23b12hvck1334yg', summary=[], type='reasoning', con